Извлечение слепых оценок GPT-4.1 mini.....


In [1]:
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'evaluations/human_and_model_validation/gpt41_mini_judging'
os.chdir(_NOTEBOOK_DIR)


In [2]:
import os
import re
from typing import Dict, List
import pandas as pd

def _read_csv(*args, **kwargs):
    frame = pd.read_csv(*args, **kwargs)
    for column in frame.select_dtypes(include=['object', 'string']).columns:
        frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
    return frame
HERE = os.path.abspath(os.getcwd())
VALIDATION = os.path.dirname(HERE)
GOLD = str(_REPOSITORY_ROOT)
NOFIX = os.path.join(GOLD, 'experiments/clinical_protocols')
BLIND_CSV = os.path.join(VALIDATION, 'blind_protocols_60.csv')
KEY_CSV = os.path.join(VALIDATION, 'blind_protocols_60_key.csv')
DELPHI_DIR = 'heterogeneous_gpt120_organizer_complete'
SOURCES: Dict[str, str] = {'delphi': os.path.join(NOFIX, 'delphi_panels', DELPHI_DIR, 'llm_as_judge', 'delphi_base_noroles_judge.csv'), 'deepseekv4': os.path.join(NOFIX, 'single_model_baselines', 'baseline_deepseekv4', 'llm_as_judge', 'baseline_noroles_deepseekv4_judge.csv'), 'gpt': os.path.join(NOFIX, 'single_model_baselines', 'baseline_gpt', 'llm_as_judge', 'baseline_noroles_gpt_judge.csv'), 'qwen': os.path.join(NOFIX, 'single_model_baselines', 'baseline_qwen', 'llm_as_judge', 'baseline_noroles_qwen_judge.csv')}
OUTPUT_CSV = os.path.join(HERE, 'blind_protocols_60_gpt41_mini_judge.csv')
STATS_CSV = os.path.join(HERE, 'blind_protocols_60_gpt41_mini_stats_by_source.csv')
print('HERE:', HERE)
for source, path in SOURCES.items():
    print(f"  {source:<12} {('OK' if os.path.exists(path) else 'НЕТ ФАЙЛА')}  {path}")


HERE: .\evaluations\human_and_model_validation\gpt41_mini_judging
  delphi       НЕТ ФАЙЛА  .\experiments/clinical_protocols\delphi_panels\heterogeneous_gpt120_organizer_complete\llm_as_judge\delphi_base_noroles_judge.csv
  deepseekv4   НЕТ ФАЙЛА  .\experiments/clinical_protocols\single_model_baselines\baseline_deepseekv4\llm_as_judge\baseline_noroles_deepseekv4_judge.csv
  gpt          НЕТ ФАЙЛА  .\experiments/clinical_protocols\single_model_baselines\baseline_gpt\llm_as_judge\baseline_noroles_gpt_judge.csv
  qwen         НЕТ ФАЙЛА  .\experiments/clinical_protocols\single_model_baselines\baseline_qwen\llm_as_judge\baseline_noroles_qwen_judge.csv


In [3]:
PROTOCOL_COLUMNS = [f'protocol_{i}' for i in range(1, 5)]
SOURCE_COLUMNS = [f'source_of_{i}' for i in range(1, 5)]

def load_blind_long(blind_csv: str, key_csv: str) -> pd.DataFrame:
    blind = _read_csv(blind_csv, sep=';', encoding='utf-8-sig', low_memory=False)
    key = _read_csv(key_csv, sep=';', encoding='utf-8-sig', low_memory=False)
    shared = list(blind.columns)
    if not blind.equals(key[shared]):
        raise ValueError('Blind CSV and key CSV differ in their shared columns')
    if len(blind) != 60 or not blind['case_id'].is_unique:
        raise ValueError('Expected exactly 60 unique blind cases')
    id_columns = [c for c in blind.columns if c not in PROTOCOL_COLUMNS]
    protocols = blind.melt(id_vars=id_columns, value_vars=PROTOCOL_COLUMNS, var_name='protocol_column', value_name='protocol')
    protocols['blind_slot'] = protocols['protocol_column'].str.extract('(\\d+)$').astype(int)
    source_key = key[['case_id', *SOURCE_COLUMNS]].melt(id_vars='case_id', value_vars=SOURCE_COLUMNS, var_name='source_column', value_name='protocol_source')
    source_key['blind_slot'] = source_key['source_column'].str.extract('(\\d+)$').astype(int)
    source_key = source_key.drop(columns='source_column')
    protocols = protocols.merge(source_key, on=['case_id', 'blind_slot'], how='left', validate='one_to_one')
    protocols = protocols.sort_values(['case_id', 'blind_slot'], kind='stable').reset_index(drop=True)
    if len(protocols) != 240 or protocols['protocol_source'].isna().any():
        raise ValueError('Expected 240 protocols with fully resolved sources')
    return protocols
blind_long = load_blind_long(BLIND_CSV, KEY_CSV)
print(f"Cases: {blind_long['case_id'].nunique()} | protocols: {len(blind_long)}")
print(pd.crosstab(blind_long['blind_slot'], blind_long['protocol_source']).to_string())


Cases: 60 | protocols: 240
protocol_source  deepseekv4  delphi  gpt  qwen
blind_slot                                    
1                        22      14   13    11
2                        10      21   13    16
3                        15       9   17    19
4                        13      16   17    14


In [4]:
EVALUATION_COLUMNS: List[str] = ['Judge_score', 'Judge_analysis', 'judge_earned', 'judge_penalties', 'judge_max', 'judge_details', 'judge_prompt_tokens', 'judge_completion_tokens', 'judge_total_tokens', 'judge_cached_calls', 'uni_details', 'uni_prompt_tokens', 'uni_completion_tokens', 'uni_total_tokens', 'uni_cached_calls']

def load_judged_source(source: str, path: str) -> pd.DataFrame:
    df = _read_csv(os.path.abspath(path), sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in df.columns:
        df = df[df['table_source'].astype(str) == 'result_df']
    df = df[df['Vignette'].notna() & df['winner_protocol'].notna()].copy()
    required = ['Vignette', 'winner_protocol', *EVALUATION_COLUMNS]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f'Missing columns in {path}: {missing}')
    if not df['Vignette'].is_unique:
        raise ValueError(f'Vignette is not unique in {path}')
    result = df[required].copy()
    result = result.rename(columns={'winner_protocol': 'judged_protocol'})
    result['protocol_source'] = source
    result['source_judge_csv'] = os.path.relpath(path, GOLD).replace(chr(92), '/')
    return result
judged = pd.concat([load_judged_source(source, path) for source, path in SOURCES.items()], ignore_index=True)
print(judged.groupby('protocol_source').size().to_string())


protocol_source
deepseekv4    228
delphi        229
gpt           229
qwen          229


In [5]:
def normalize_protocol(text: str) -> str:
    value = '' if pd.isna(text) else str(text)
    value = value.replace('\r\n', '\n').replace('\r', '\n')
    value = re.split('(?im)^\\s*Protocol Author\\s*:', value, maxsplit=1)[0]
    return value.rstrip()
extended = blind_long.merge(judged, on=['Vignette', 'protocol_source'], how='left', validate='many_to_one')
extended['protocol_text_matches_judged_source'] = extended['protocol'].map(normalize_protocol) == extended['judged_protocol'].map(normalize_protocol)
missing_scores = int(extended['Judge_score'].isna().sum())
text_mismatches = int((~extended['protocol_text_matches_judged_source']).sum())
missing_comments = int(extended['Judge_analysis'].fillna('').astype(str).str.strip().eq('').sum())
print(f'Rows: {len(extended)}')
print(f'Missing Judge_score: {missing_scores}')
print(f'Missing Judge_analysis: {missing_comments}')
print(f'Protocol text mismatches: {text_mismatches}')
if len(extended) != 240:
    raise AssertionError(f'Expected 240 rows, got {len(extended)}')
if missing_scores or missing_comments or text_mismatches:
    raise ValueError('Extraction validation failed; output was not saved')


Rows: 240
Missing Judge_score: 0
Missing Judge_analysis: 0
Protocol text mismatches: 0


In [6]:
SOURCE_ORDER = ['delphi', 'deepseekv4', 'gpt', 'qwen']
statistics = extended.groupby('protocol_source', sort=False)['Judge_score'].agg(n='count', mean='mean', std='std', median='median', min='min', max='max').reindex(SOURCE_ORDER).reset_index()
for column in ['mean', 'std', 'median', 'min', 'max']:
    statistics[column] = statistics[column].round(4)
output_columns = ['case_id', 'Vignette', 'case_text', 'Criterion', 'Diagnosis_vulgar', 'Type', 'Genealogy', 'Sub Clinical Specialties', 'Age Group', 'blind_slot', 'protocol_column', 'protocol_source', 'protocol', *EVALUATION_COLUMNS, 'protocol_text_matches_judged_source', 'source_judge_csv']
extended_output = extended[output_columns].sort_values(['case_id', 'blind_slot'], kind='stable').reset_index(drop=True)
extended_output.to_csv(OUTPUT_CSV, sep=';', index=False, encoding='utf-8-sig')
statistics.to_csv(STATS_CSV, sep=';', index=False, encoding='utf-8-sig')
print('Judge_score by protocol source:')
print(statistics.to_string(index=False))
print(f'\nSaved judgments: {OUTPUT_CSV} ({len(extended_output)} rows)')
print(f'Saved statistics: {STATS_CSV}')


Judge_score by protocol source:
protocol_source  n   mean    std  median    min    max
         delphi 60 0.7705 0.1227  0.7799 0.5000 1.0000
     deepseekv4 60 0.6232 0.1774  0.6380 0.1974 1.0000
            gpt 60 0.5851 0.1893  0.6033 0.0952 0.9423
           qwen 60 0.5570 0.1715  0.5755 0.1605 0.9344

Saved judgments: .\evaluations\human_and_model_validation\gpt41_mini_judging\blind_protocols_60_gpt41_mini_judge.csv (240 rows)
Saved statistics: .\evaluations\human_and_model_validation\gpt41_mini_judging\blind_protocols_60_gpt41_mini_stats_by_source.csv
